# Closed-loop trajectory-litho domain engineering — v2

Builds on `Trajectory_based_domain_writting_v1.ipynb`. Full loop:

**(1) acquire PFM → (2) extract domains / polarization → (3) score against a target → (4) generate the next trajectorylitho path → (5) write → repeat.**

Organization:
1. Config & microscope interface (`aespm` + `TL_*` panel, same conventions as v1)
2. Trajectory toolkit (`TrajectoryBuilder`, line / rose / raster-fill / spiral / flower generators, target **compiler**)
3. PFM analysis (domain labels, vector PFM, drift registration, features)
4. Targets & objective
5. Dry-run simulator (debug the whole loop with **no microscope**)
6. Orchestration (`run_iteration`) + GP-EI active learner
7. **Session A** — compass-rose calibration → sector map
8. **Session B** — compiler test (bicrystal / designed wall)
9. **Session C** — Bayesian optimization toward a vortex / convergent target

Anything controller-specific I couldn't verify is marked `# >>> PLACEHOLDER` — search for that string. Set `CONFIG["DRY_RUN"] = False` at the microscope; with `True` the whole notebook runs offline against a toy forward model.

## 0. Imports & session config

In [ ]:
import os, json, time
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

try:
    import aespm as ae
    HAVE_AESPM = True
except ImportError:
    HAVE_AESPM = False
    print("aespm not found -> hardware calls disabled, DRY_RUN forced True")

In [ ]:
CONFIG = dict(
    # --- paths -----------------------------------------------------------
    data_folder = r"C:\Users\Asylum User\Documents\Asylum Research Data\260612\PZTO-111",
    work_dir    = "trajlitho_session",      # trajectories + logs live here

    # --- microscope / litho ------------------------------------------------
    DRY_RUN       = True,    # False at the microscope
    field_um      = 5.0,     # litho field size (square, corner origin)
    write_speed_ums = 1.0,   # tip speed during litho
    step_um       = 0.02,    # point spacing along trajectory (sets dose uniformity)
    travel_v      = 0.0,     # bias during repositioning strokes
    litho_margin_s = 10.0,   # extra wait after estimated litho duration

    # --- PFM readout ---------------------------------------------------------
    chan_height = 0, chan_amp = 1, chan_phase = 3,   # ibw channel indices (as in v1)
    scan_px     = 256,

    # --- material: PZTO(111), three easy axes at 120 deg --------------------
    # axis0_offset_deg is the lab-frame angle of the first easy axis;
    # you will refine it from the pristine stripe orientations / Session A.
    axis0_offset_deg = 0.0,
    n_variants  = 6,
    easy_axes_deg = None,    # filled below

    # --- switching physics (seed values; Session A refines) -----------------
    v_coercive_guess = 4.0,  # |V| onset (Checa: 4-6 V in PSTO; measure yours)
    v_write          = 6.0,
    # bias sign convention: +1 if positive tip bias selects the variant
    # PARALLEL to tip motion, -1 if anti-parallel (Balke-style). Session A decides.
    bias_sign_convention = -1,
    linewidth_um_guess   = 0.12,   # written track width (from first line tests)
)
CONFIG["easy_axes_deg"] = [CONFIG["axis0_offset_deg"] + k*60 for k in range(6)]
os.makedirs(CONFIG["work_dir"], exist_ok=True)
if not HAVE_AESPM:
    CONFIG["DRY_RUN"] = True
print(json.dumps({k: v for k, v in CONFIG.items() if k != "easy_axes_deg"}, indent=1))
print("easy axes:", CONFIG["easy_axes_deg"])

## 1. Microscope interface

Same conventions as v1: `ae.Experiment`, `ae.write_spm`, the `TL_LoadBuildPy / TL_RunPy / TL_StopPy` panel commands, and the trajectory file format (3 tab-separated columns `X_m  Y_m  V`, metres, corner origin).

In [ ]:
if HAVE_AESPM and not CONFIG["DRY_RUN"]:
    exp = ae.Experiment(folder=CONFIG["data_folder"])

    def load_ibw(self, lines=False):
        fname = ae.get_files(path=self.folder, client=self.client)[0]
        return ae.tools.load_ibw(fname)
    exp.add_func(load_ibw)

    def check_files(self, wait=360):
        return ae.check_file_number(path=self.folder, retry=int(wait/0.1))
    exp.add_func(check_files)
else:
    exp = None  # dry run

In [ ]:
def send_trajectory(txt_path):
    """Load+build a trajectory file in the TL panel (Igor needs doubled backslashes)."""
    full = os.path.abspath(txt_path)
    igor_path = full.replace("\\", "\\\\")
    ae.write_spm(commands=f'TL_LoadBuildPy("{igor_path}")')

def start_trajectory(speed_ums=None, center_um=None):
    s = CONFIG["write_speed_ums"] if speed_ums is None else speed_ums
    if center_um is None:
        ae.write_spm(commands=f'TL_RunPy({s}, 0, 0, 0)')          # live scan center
    else:
        ae.write_spm(commands=f'TL_RunPy({s}, 1, {center_um[0]}, {center_um[1]})')

def stop_trajectory():
    ae.write_spm(commands='TL_StopPy()')

def litho_is_running():
    """# >>> PLACEHOLDER: read GV("LithoRunning") back into Python.
    v1 only shows `print GV("LithoRunning")` on the Igor side. If aespm can
    round-trip a value (e.g. ae.read_spm, or writing GV(...) into a buffer
    ibw like your read_meter() does with Meter.ibw), implement it here and
    wait_for_litho() will switch from time-estimate to true polling."""
    raise NotImplementedError

def wait_for_litho(tb, speed_ums=None, poll=False):
    """Block until the litho run finishes.
    Default: sleep for path_length/speed * 1.15 + margin (always works).
    poll=True uses litho_is_running() once you implement it."""
    s = CONFIG["write_speed_ums"] if speed_ums is None else speed_ums
    t_est = tb.path_length_um() / s * 1.15 + CONFIG["litho_margin_s"]
    print(f"litho ETA ~{t_est:.0f} s")
    if CONFIG["DRY_RUN"]:
        return
    if poll:
        t0 = time.time()
        while litho_is_running():
            time.sleep(1)
            if time.time() - t0 > 3 * t_est:
                print("WARNING: litho poll timeout"); break
    else:
        time.sleep(t_est)

In [ ]:
def acquire_pfm():
    """Trigger one PFM scan and return (height, amplitude, phase_deg) arrays.

    # >>> PLACEHOLDER: programmatic scan-frame setup. To guarantee the scan
    # frame covers the litho field at the right size/offset, set ScanSize /
    # offsets / points-and-lines here via aespm before DownScan, e.g. the
    # aespm parameter actions or ae.write_spm with the appropriate Igor
    # PV/td_WriteValue commands for your build. Until then: set the scan
    # frame manually in the AR software so it matches CONFIG["field_um"].
    """
    if CONFIG["DRY_RUN"]:
        raise RuntimeError("acquire_pfm() called in DRY_RUN mode")
    exp.execute('DownScan', wait=2)
    exp.check_files(wait=600)
    w = exp.load_ibw()
    return (w.data[CONFIG["chan_height"]],
            w.data[CONFIG["chan_amp"]],
            w.data[CONFIG["chan_phase"]])

def acquire_pfm_second_orientation():
    """# >>> PLACEHOLDER: second in-plane sensing axis for full vector PFM.
    Six variants cannot be resolved from ONE lateral channel (you only get a
    projection). Options: (a) physically rotate the sample 90 deg and rescan
    (manual; prompt the user), (b) torsional vs buckling channels, (c) two
    cantilever orientations. Implement whichever your setup supports; return
    (height, amp, phase_deg) like acquire_pfm()."""
    raise NotImplementedError

## 2. Trajectory toolkit

Everything below is pure Python (tested offline). Coordinates in µm, corner origin `0..field_um`, saved in metres — identical to the v1 spiral generator's convention. Two physics-aware details:

* **Constant arc-length resampling** — the TL panel steps through points at a fixed rate, so constant point spacing ⇒ constant tip speed and uniform bias dose.
* **Unidirectional strokes** — motion direction is the variant-selection knob (Balke 2009, Checa 2024), so region fills run every biased stroke in the *same* direction with zero-bias travel returns, instead of boustrophedon.

In [ ]:
# ----------------------------------------------------------------------
# Low-level helpers
# ----------------------------------------------------------------------
def resample_constant_step(x_um, y_um, v, step_um):
    """Resample a polyline to constant arc-length step.

    The TL panel steps through points at a fixed rate, so constant point
    spacing => constant tip speed and constant bias dose per unit length.
    """
    x_um, y_um, v = map(np.asarray, (x_um, y_um, v))
    d = np.hypot(np.diff(x_um), np.diff(y_um))
    s = np.concatenate([[0.0], np.cumsum(d)])
    if s[-1] == 0:
        return x_um[:1], y_um[:1], np.atleast_1d(v)[:1]
    n = max(2, int(np.ceil(s[-1] / step_um)) + 1)
    si = np.linspace(0, s[-1], n)
    return (np.interp(si, s, x_um), np.interp(si, s, y_um),
            np.interp(si, s, np.broadcast_to(v, x_um.shape)))


class TrajectoryBuilder:
    """Accumulate biased strokes; insert zero-bias travel moves between them.

    All coordinates in micrometres, corner origin (0..field_um), matching the
    convention of generate_spiral_trajectory in the original notebook.
    """

    def __init__(self, field_um=20.0, step_um=0.02, travel_v=0.0,
                 nan_breaks=False):
        self.field_um = field_um
        self.step_um = step_um
        self.travel_v = travel_v   # bias during repositioning (keep 0!)
        self.nan_breaks = nan_breaks  # PLACEHOLDER: only if TL supports NaN pen-up
        self._segs = []            # list of (x_um, y_um, v) arrays

    def stroke(self, pts_um, v):
        """Add a biased stroke. pts_um: (N,2) array; v: scalar or (N,) array."""
        pts_um = np.asarray(pts_um, float)
        x, y = pts_um[:, 0], pts_um[:, 1]
        x, y, vv = resample_constant_step(x, y, v, self.step_um)
        self._segs.append((x, y, vv))
        return self

    def to_arrays(self):
        """Concatenate strokes, inserting travel moves at travel_v."""
        xs, ys, vs = [], [], []
        for i, (x, y, v) in enumerate(self._segs):
            if i > 0:
                if self.nan_breaks:
                    xs.append([np.nan]); ys.append([np.nan]); vs.append([np.nan])
                else:
                    x0, y0 = xs[-1][-1], ys[-1][-1]
                    tx, ty, tv = resample_constant_step(
                        [x0, x[0]], [y0, y[0]], self.travel_v, self.step_um)
                    xs.append(tx); ys.append(ty); vs.append(tv)
            xs.append(x); ys.append(y); vs.append(v)
        return (np.concatenate(xs), np.concatenate(ys), np.concatenate(vs))

    def save(self, filename, include_header=True):
        x_um, y_um, v = self.to_arrays()
        data = np.column_stack([x_um * 1e-6, y_um * 1e-6, v])
        header = "X_m\tY_m\tV" if include_header else ""
        np.savetxt(filename, data, delimiter="\t", header=header,
                   comments="# ", fmt="%.10g")
        finite = np.isfinite(x_um)
        d = np.hypot(np.diff(x_um[finite]), np.diff(y_um[finite]))
        print(f"Wrote {len(x_um)} pts to '{filename}'  "
              f"(path length {np.nansum(d):.1f} um, "
              f"{len(self._segs)} strokes)")
        return x_um * 1e-6, y_um * 1e-6, v

    def path_length_um(self):
        x, y, _ = self.to_arrays()
        m = np.isfinite(x)
        return float(np.nansum(np.hypot(np.diff(x[m]), np.diff(y[m]))))

In [ ]:
# ----------------------------------------------------------------------
# Pattern generators (all return a TrajectoryBuilder)
# ----------------------------------------------------------------------
def gen_line(angle_deg, length_um, center_um, v, field_um=20.0, step_um=0.02,
             tb=None):
    """One straight biased stroke through center_um along angle_deg."""
    tb = tb or TrajectoryBuilder(field_um, step_um)
    a = np.deg2rad(angle_deg)
    dx, dy = np.cos(a), np.sin(a)
    cx, cy = center_um
    p0 = (cx - dx * length_um / 2, cy - dy * length_um / 2)
    p1 = (cx + dx * length_um / 2, cy + dy * length_um / 2)
    tb.stroke(np.array([p0, p1]), v)
    return tb


def gen_compass_rose(angles_deg, biases_v, line_len_um=1.0, cell_um=2.0,
                     margin_um=1.0, step_um=0.02, repeats=1):
    """Calibration pattern: grid of single strokes, one per (angle, bias).

    Returns (TrajectoryBuilder, manifest) where manifest is a list of dicts
    {row, col, cx, cy, angle_deg, V} for later lookup against the PFM image.
    Rows = biases, cols = angles (x repeats).
    """
    angles = list(angles_deg) * repeats
    ncol, nrow = len(angles), len(biases_v)
    field = ncol * cell_um + 2 * margin_um
    field = max(field, nrow * cell_um + 2 * margin_um)
    tb = TrajectoryBuilder(field_um=field, step_um=step_um)
    manifest = []
    for r, V in enumerate(biases_v):
        for c, ang in enumerate(angles):
            cx = margin_um + (c + 0.5) * cell_um
            cy = margin_um + (r + 0.5) * cell_um
            gen_line(ang, line_len_um, (cx, cy), V, tb=tb)
            manifest.append(dict(row=r, col=c, cx_um=cx, cy_um=cy,
                                 angle_deg=ang % 360, V=V))
    return tb, manifest


def gen_raster_fill(mask, extent_um, stroke_angle_deg, pitch_um, v,
                    step_um=0.02, tb=None, unidirectional=True):
    """Fill the True region of `mask` with parallel strokes.

    mask        : (H, W) boolean image, row 0 at y=0 (origin='lower').
    extent_um   : (x0, x1, y0, y1) of the mask in litho coordinates.
    stroke_angle_deg : direction of tip MOTION during biased strokes.
                  This is the physics knob: motion direction selects the
                  ferroelastic variant (Balke 2009 / Checa 2024).
    unidirectional : if True every stroke runs in the same direction
                  (+angle), with zero-bias travel back. Boustrophedon would
                  alternate the effective field direction line-to-line.
    """
    tb = tb or TrajectoryBuilder(field_um=max(extent_um[1], extent_um[3]),
                                 step_um=step_um)
    mask = np.asarray(mask, bool)
    H, W = mask.shape
    x0, x1, y0, y1 = extent_um
    a = np.deg2rad(stroke_angle_deg)
    u = np.array([np.cos(a), np.sin(a)])      # along stroke
    n = np.array([-np.sin(a), np.cos(a)])     # stroke-normal

    # corners projected on normal -> range of scan lines
    corners = np.array([[x0, y0], [x1, y0], [x0, y1], [x1, y1]])
    tmin, tmax = min(corners @ n), max(corners @ n)
    smin, smax = min(corners @ u), max(corners @ u)

    def inside(pts):
        ix = np.clip(((pts[:, 0] - x0) / (x1 - x0) * W).astype(int), 0, W - 1)
        iy = np.clip(((pts[:, 1] - y0) / (y1 - y0) * H).astype(int), 0, H - 1)
        in_box = ((pts[:, 0] >= x0) & (pts[:, 0] <= x1) &
                  (pts[:, 1] >= y0) & (pts[:, 1] <= y1))
        return in_box & mask[iy, ix]

    nstrokes = 0
    for t in np.arange(tmin + pitch_um / 2, tmax, pitch_um):
        s = np.arange(smin, smax + step_um, step_um)
        pts = np.outer(s, u) + t * n
        ok = inside(pts)
        if not ok.any():
            continue
        # split into contiguous runs
        idx = np.flatnonzero(ok)
        breaks = np.flatnonzero(np.diff(idx) > 1)
        runs = np.split(idx, breaks + 1)
        for run in runs:
            if len(run) < 2:
                continue
            seg = pts[run]
            if (not unidirectional) and (nstrokes % 2 == 1):
                seg = seg[::-1]
            tb.stroke(seg, v)
            nstrokes += 1
    return tb


def gen_spiral(turns=20, max_radius_um=2.0, center_um=(0, 0), v=6.0,
               direction='out', chirality='ccw', step_um=0.02, field_um=20.0,
               tb=None):
    """Archimedean spiral with CONSTANT bias (variant-writing version of the
    original sinusoidal-bias generator). direction: 'out' (centre->edge,
    radial slow direction outward) or 'in'. chirality: 'ccw' / 'cw'."""
    tb = tb or TrajectoryBuilder(field_um=field_um, step_um=step_um)
    th_max = turns * 2 * np.pi
    # parametrize finely, then arc-length resampling in stroke() fixes spacing
    th = np.linspace(0, th_max, int(50 * turns) + 2)
    r = max_radius_um * th / th_max
    if direction == 'in':
        r = r[::-1]
    sgn = 1.0 if chirality == 'ccw' else -1.0
    x = center_um[0] + r * np.cos(sgn * th)
    y = center_um[1] + r * np.sin(sgn * th)
    tb.stroke(np.column_stack([x, y]), v)
    return tb


def gen_flower(petals=24, radius_um=2.0, center_um=(0, 0), v=-6.0,
               chirality='ccw', step_um=0.02, field_um=20.0, tb=None,
               bias_on_outward_only=False):
    """Flower scan: repeated radial out-and-back strokes, advancing in angle.
    Radial fast-scan direction (cf. Checa Fig. 5)."""
    tb = tb or TrajectoryBuilder(field_um=field_um, step_um=step_um)
    sgn = 1.0 if chirality == 'ccw' else -1.0
    cx, cy = center_um
    for i in range(petals):
        a = sgn * 2 * np.pi * i / petals
        tip = (cx + radius_um * np.cos(a), cy + radius_um * np.sin(a))
        tb.stroke(np.array([(cx, cy), tip]), v)                 # outward
        v_back = 0.0 if bias_on_outward_only else v
        tb.stroke(np.array([tip, (cx, cy)]), v_back)            # back
    return tb

In [ ]:
# ----------------------------------------------------------------------
# The "compiler": target label image -> trajectory
# ----------------------------------------------------------------------
def compile_target(target_labels, extent_um, sector_map, v_map,
                   pitch_um=0.05, step_um=0.02, order='area_desc'):
    """Translate a target variant map into a write program.

    target_labels : (H, W) int image of desired variant index (-1 = leave).
    sector_map    : {variant: stroke_angle_deg} from the compass-rose
                    calibration (motion direction that yields that variant).
    v_map         : {variant: bias_V}.
    order         : 'area_desc' writes large regions (background) first so
                    fine features written later are not overwritten.
    """
    target_labels = np.asarray(target_labels)
    variants = [v for v in np.unique(target_labels) if v >= 0]
    if order == 'area_desc':
        variants.sort(key=lambda u: -(target_labels == u).sum())
    x1 = max(extent_um[1], extent_um[3])
    tb = TrajectoryBuilder(field_um=x1, step_um=step_um)
    for u in variants:
        gen_raster_fill(target_labels == u, extent_um,
                        stroke_angle_deg=sector_map[u],
                        pitch_um=pitch_um, v=v_map[u], step_um=step_um, tb=tb)
    return tb

In [ ]:
def visualize_trajectory(tb=None, x_m=None, y_m=None, v=None, *, filename=None,
                         field_um=None, title=None):
    """Same as v1, plus accepts a TrajectoryBuilder directly."""
    if tb is not None:
        x, y, v = tb.to_arrays(); x_m, y_m = x*1e-6, y*1e-6
        field_um = field_um or tb.field_um
    if filename is not None:
        d = np.loadtxt(filename); x_m, y_m, v = d[:,0], d[:,1], d[:,2]
        title = title or f"Trajectory from '{filename}'"
    x_um, y_um, v = np.asarray(x_m)*1e6, np.asarray(y_m)*1e6, np.asarray(v)
    finite = np.isfinite(x_um) & np.isfinite(y_um); idx = np.arange(len(x_um))
    fig = plt.figure(figsize=(13, 5.5)); gs = fig.add_gridspec(2,2,width_ratios=[1.3,1])
    ax0 = fig.add_subplot(gs[:,0])
    sc = ax0.scatter(x_um[finite], y_um[finite], c=v[finite], cmap="rainbow", s=6, zorder=3)
    ax0.plot(x_um, y_um, color="0.8", lw=.5, zorder=1)
    f = idx[finite]
    if len(f):
        ax0.scatter([x_um[f[0]]],[y_um[f[0]]],s=120,marker="o",facecolors="none",
                    edgecolors="g",lw=2,zorder=4,label="start")
        ax0.scatter([x_um[f[-1]]],[y_um[f[-1]]],s=120,marker="s",facecolors="none",
                    edgecolors="r",lw=2,zorder=4,label="end")
    fu = field_um or CONFIG["field_um"]
    ax0.add_patch(plt.Rectangle((0,0), fu, fu, fill=False, ls="--", ec="k"))
    ax0.set_aspect("equal"); ax0.set_xlabel("X (um)"); ax0.set_ylabel("Y (um)")
    ax0.set_title(title or "Trajectory (color = bias)"); ax0.legend(fontsize=8)
    fig.colorbar(sc, ax=ax0, label="Bias (V)", fraction=.046, pad=.04)
    ax1 = fig.add_subplot(gs[0,1]); ax1.plot(idx, v, lw=1)
    ax1.set_xlabel("point index"); ax1.set_ylabel("Bias (V)"); ax1.grid(alpha=.3)
    ax2 = fig.add_subplot(gs[1,1])
    step = np.hypot(np.diff(x_um), np.diff(y_um))
    ax2.plot(step, lw=1, color="C1"); ax2.set_xlabel("point index")
    ax2.set_ylabel("step (um)"); ax2.grid(alpha=.3)
    plt.tight_layout(); plt.show()

## 3. PFM analysis → domain labels & features

With **one** lateral channel you can only separate 2 (or a few) clusters along the sensing projection (`extract_domains_1ch`). Resolving all 6 variants of the 120°-axis film needs a second sensing axis → `vector_pfm` + `snap_to_axes` (second acquisition is a placeholder until you wire it up).

In [ ]:
# # Domain extraction
def complex_pr(amp, phase_deg):
    return np.asarray(amp) * np.exp(1j * np.deg2rad(np.asarray(phase_deg)))


def extract_domains_1ch(amp, phase_deg, n_clusters=2, seed=0):
    """Cluster one lateral-PFM channel into n_clusters using GMM on
    (Re, Im) of the complex piezoresponse. Returns (labels, means)."""
    from sklearn.mixture import GaussianMixture
    pr = complex_pr(amp, phase_deg)
    X = np.column_stack([pr.real.ravel(), pr.imag.ravel()])
    X = (X - X.mean(0)) / (X.std(0) + 1e-12)
    gm = GaussianMixture(n_clusters, random_state=seed, n_init=3).fit(X)
    return gm.predict(X).reshape(np.shape(amp)), gm.means_


def vector_pfm(pr_a, pr_b, axis_a_deg, axis_b_deg):
    """Combine two LPFM channels measured along two in-plane sensing axes
    (two cantilever / sample orientations) into (Px, Py) maps.
    pr_a, pr_b: signed responses, e.g. Re of drift-registered complex PR."""
    A = np.array([[np.cos(np.deg2rad(axis_a_deg)), np.sin(np.deg2rad(axis_a_deg))],
                  [np.cos(np.deg2rad(axis_b_deg)), np.sin(np.deg2rad(axis_b_deg))]])
    Ainv = np.linalg.inv(A)
    P = np.einsum('ij,jhw->ihw', Ainv,
                  np.stack([np.asarray(pr_a), np.asarray(pr_b)]))
    return P[0], P[1]   # Px, Py


def snap_to_axes(px, py, easy_axes_deg):
    """Assign each pixel to the nearest easy-axis variant.
    easy_axes_deg: e.g. a0 + [0,60,120,180,240,300] for the 120-deg film.
    Returns int label map (index into easy_axes_deg)."""
    ax = np.deg2rad(np.asarray(easy_axes_deg))
    dots = (np.cos(ax)[:, None, None] * px[None] +
            np.sin(ax)[:, None, None] * py[None])
    return np.argmax(dots, axis=0)

In [ ]:
def register_offset(img_ref, img, px_per_um):
    """Drift between two scans via phase cross-correlation -> (dx_um, dy_um).
    # >>> PLACEHOLDER: apply the offset. Two options:
    #  (a) shift the litho center passed to start_trajectory(center_um=...)
    #  (b) shift the analysis extent when comparing to the target.
    Either works; (a) keeps writes registered to earlier writes."""
    from skimage.registration import phase_cross_correlation
    shift, _, _ = phase_cross_correlation(np.asarray(img_ref), np.asarray(img),
                                          upsample_factor=10)
    dy_px, dx_px = shift
    return dx_px / px_per_um, dy_px / px_per_um

In [ ]:
# # Features / objective
def variant_fractions(labels, n_variants):
    return np.bincount(np.ravel(labels), minlength=n_variants) / labels.size


def winding_number(px, py, center_px=None, radius_frac=0.3):
    """Topological winding of the in-plane polarization angle on a circle
    around center. +1 for vortex/convergent/divergent, 0 for uniform."""
    H, W = px.shape
    cy, cx = (H // 2, W // 2) if center_px is None else center_px
    R = radius_frac * min(H, W) / 2
    th = np.linspace(0, 2 * np.pi, 360, endpoint=False)
    iy = np.clip((cy + R * np.sin(th)).astype(int), 0, H - 1)
    ix = np.clip((cx + R * np.cos(th)).astype(int), 0, W - 1)
    ang = np.unwrap(np.arctan2(py[iy, ix], px[iy, ix]))
    # close the loop (first sample repeated at the end) and count turns
    closing = np.angle(np.exp(1j * (ang[0] - ang[-1])))
    return float((ang[-1] + closing - ang[0]) / (2 * np.pi))


def mean_iou(labels, target, n_variants):
    ious = []
    for u in range(n_variants):
        a, b = labels == u, target == u
        if not (a.any() or b.any()):
            continue
        inter, union = (a & b).sum(), (a | b).sum()
        ious.append(inter / union if union else 1.0)
    return float(np.mean(ious)) if ious else 0.0


def score_against_target(labels, target, n_variants,
                         px=None, py=None, target_winding=None,
                         w_iou=1.0, w_wind=0.5):
    """Scalar loss in [0, ~2]; lower is better."""
    loss = w_iou * (1.0 - mean_iou(labels, target, n_variants))
    if target_winding is not None and px is not None:
        loss += w_wind * abs(winding_number(px, py) - target_winding)
    return loss

In [ ]:
# # Target patterns (label images)
def target_uniform(shape, variant):
    return np.full(shape, variant, int)


def target_bicrystal(shape, v_left, v_right, wall_angle_deg=90.0):
    H, W = shape
    yy, xx = np.mgrid[0:H, 0:W]
    n = np.array([np.cos(np.deg2rad(wall_angle_deg + 90)),
                  np.sin(np.deg2rad(wall_angle_deg + 90))])
    side = (xx - W / 2) * n[0] + (yy - H / 2) * n[1]
    return np.where(side < 0, v_left, v_right).astype(int)


def target_vortex(shape, easy_axes_deg, sense=+1, radial=False, bg=-1,
                  radius_frac=0.45):
    """Six-sector vortex (sense=+1 ccw / -1 cw) or convergent/divergent
    (radial=True, sense=+1 divergent / -1 convergent) snapped to easy axes."""
    H, W = shape
    yy, xx = np.mgrid[0:H, 0:W]
    dx, dy = xx - W / 2, yy - H / 2
    r = np.hypot(dx, dy)
    phi = np.arctan2(dy, dx)
    pol = phi if radial else phi + np.pi / 2 * np.sign(sense)
    if radial and sense < 0:
        pol = phi + np.pi
    px, py = np.cos(pol), np.sin(pol)
    lab = snap_to_axes(px, py, easy_axes_deg)
    lab[r > radius_frac * min(H, W)] = bg
    return lab

## 4. Dry-run simulator

A deliberately crude forward model — paints the nearest-easy-axis variant along the biased path wherever |V| ≥ V_c — so every downstream cell (compiler, scoring, active learning) can be debugged end-to-end **without the microscope**. It is *not* physics; replace observations with real PFM the moment `DRY_RUN=False`.

In [ ]:
# # Dry-run simulator: toy forward model trajectory -> label image
def simulate_write(x_m, y_m, v, extent_um, shape, easy_axes_deg,
                   initial_labels=None, v_coercive=4.0, linewidth_um=0.12,
                   bias_sign_convention=-1):
    """Paint the variant selected by local motion direction wherever
    |V| >= v_coercive. Nearest-easy-axis rule; negative bias selects the
    axis anti-parallel to motion if bias_sign_convention=-1 (Balke-like).
    Crude, but exercises the full software loop end-to-end."""
    H, W = shape
    x0, x1, y0, y1 = extent_um
    lab = (np.zeros(shape, int) if initial_labels is None
           else initial_labels.copy())
    xu, yu = np.asarray(x_m) * 1e6, np.asarray(y_m) * 1e6
    ax = np.deg2rad(np.asarray(easy_axes_deg))
    rad_px = max(1, int(linewidth_um / 2 / ((x1 - x0) / W)))
    dyy, dxx = np.mgrid[-rad_px:rad_px + 1, -rad_px:rad_px + 1]
    disk = dxx**2 + dyy**2 <= rad_px**2
    for i in range(1, len(xu)):
        if not np.isfinite(xu[i] + xu[i - 1]):
            continue
        if abs(v[i]) < v_coercive:
            continue
        mdir = np.arctan2(yu[i] - yu[i - 1], xu[i] - xu[i - 1])
        if np.sign(v[i]) * bias_sign_convention < 0:
            mdir += np.pi
        u = int(np.argmax(np.cos(ax - mdir)))
        cx = int((xu[i] - x0) / (x1 - x0) * W)
        cy = int((yu[i] - y0) / (y1 - y0) * H)
        ys, xs = np.nonzero(disk)
        ys, xs = ys + cy - rad_px, xs + cx - rad_px
        ok = (ys >= 0) & (ys < H) & (xs >= 0) & (xs < W)
        lab[ys[ok], xs[ok]] = u
    return lab


def labels_to_vectors(labels, easy_axes_deg):
    ax = np.deg2rad(np.asarray(easy_axes_deg))
    return np.cos(ax)[labels], np.sin(ax)[labels]

## 5. Active learner (GP surrogate + Expected Improvement)

In [ ]:
# # GP + Expected Improvement active learner
class ActiveLearner:
    """Minimal Bayesian-optimization loop over a box parameter space.

    space: {name: (low, high)} for continuous, {name: [choices]} for
    categorical (encoded one-hot internally is overkill here; we encode
    categoricals as integer levels and round on decode).
    Minimizes the loss returned by your run function.
    """

    def __init__(self, space, seed=0, log_path="al_log.json"):
        self.space = space
        self.names = list(space)
        self.rng = np.random.default_rng(seed)
        self.X, self.y, self.meta = [], [], []
        self.log_path = log_path

    # ---- encoding -----------------------------------------------------
    def _bounds(self):
        lo, hi = [], []
        for n in self.names:
            s = self.space[n]
            if isinstance(s, (list, tuple)) and not np.isscalar(s[0]):
                raise ValueError
            if isinstance(s, list):           # categorical
                lo.append(0); hi.append(len(s) - 1e-9)
            else:
                lo.append(s[0]); hi.append(s[1])
        return np.array(lo), np.array(hi)

    def decode(self, x):
        p = {}
        for n, xi in zip(self.names, x):
            s = self.space[n]
            p[n] = s[int(xi)] if isinstance(s, list) else float(xi)
        return p

    # ---- seeding & suggestion -----------------------------------------
    def seed_points(self, n):
        from scipy.stats import qmc
        lo, hi = self._bounds()
        u = qmc.LatinHypercube(d=len(lo), seed=int(self.rng.integers(1e6))
                               ).random(n)
        return [self.decode(lo + u_i * (hi - lo)) for u_i in u]

    def suggest(self, n_candidates=4096, xi=0.01):
        if len(self.y) < 3:
            return self.seed_points(1)[0]
        from sklearn.gaussian_process import GaussianProcessRegressor
        from sklearn.gaussian_process.kernels import Matern, WhiteKernel
        from scipy.stats import norm
        lo, hi = self._bounds()
        X = (np.array(self.X) - lo) / (hi - lo)
        y = np.array(self.y)
        gp = GaussianProcessRegressor(
            Matern(nu=2.5, length_scale=0.3 * np.ones(X.shape[1])) +
            WhiteKernel(1e-3), normalize_y=True, n_restarts_optimizer=3,
            random_state=0).fit(X, y)
        C = self.rng.random((n_candidates, X.shape[1]))
        mu, sd = gp.predict(C, return_std=True)
        best = y.min()
        z = (best - xi - mu) / np.maximum(sd, 1e-9)
        ei = (best - xi - mu) * norm.cdf(z) + sd * norm.pdf(z)
        return self.decode(lo + C[np.argmax(ei)] * (hi - lo))

    # ---- bookkeeping ---------------------------------------------------
    def observe(self, params, loss, meta=None):
        self.X.append([params[n] if not isinstance(self.space[n], list)
                       else self.space[n].index(params[n])
                       for n in self.names])
        self.y.append(float(loss))
        self.meta.append(meta or {})
        with open(self.log_path, "w") as f:
            json.dump({"names": self.names, "X": self.X, "y": self.y,
                       "meta": self.meta}, f, indent=1, default=str)

    def best(self):
        i = int(np.argmin(self.y))
        return self.decode(self.X[i]), self.y[i]

## 6. Orchestration: one closed-loop iteration

`params → trajectory → write → scan → labels → loss`. Parameter families: `"spiral"`, `"flower"`, `"compiler"`. In hardware mode the label extraction defaults to 1-channel GMM with `n_variants` clusters — fine for scoring simple targets; switch to vector PFM (two orientations) when you need unambiguous 6-variant maps.

In [ ]:
ITER_STATE = {"k": 0, "last_scan": None}

def params_to_builder(family, p):
    fld = CONFIG["field_um"]; ctr = (fld/2, fld/2); st = CONFIG["step_um"]
    if family == "spiral":
        return gen_spiral(turns=int(round(p["turns"])),
                          max_radius_um=p["radius_um"], center_um=ctr,
                          v=p["v"], direction=p.get("direction", "out"),
                          chirality=p.get("chirality", "ccw"),
                          step_um=st, field_um=fld)
    if family == "flower":
        return gen_flower(petals=int(round(p["petals"])),
                          radius_um=p["radius_um"], center_um=ctr, v=p["v"],
                          chirality=p.get("chirality", "ccw"), step_um=st,
                          field_um=fld,
                          bias_on_outward_only=bool(p.get("outward_only", 1)))
    if family == "compiler":
        # p must carry: target_labels, sector_map, v_map, pitch_um
        return compile_target(p["target_labels"], (0, fld, 0, fld),
                              p["sector_map"], p["v_map"],
                              pitch_um=p["pitch_um"], step_um=st)
    raise ValueError(family)

def measure_labels(target_shape):
    """Acquire PFM and return (labels, px, py). Hardware path uses 1-channel
    GMM unless you implement the second orientation."""
    h, a, ph = acquire_pfm()
    labels, _ = extract_domains_1ch(a, ph, n_clusters=CONFIG["n_variants"])
    # >>> PLACEHOLDER: with vector PFM, replace by:
    #   h2, a2, ph2 = acquire_pfm_second_orientation()
    #   pra = (complex_pr(a, ph)).real;  prb = (complex_pr(a2, ph2)).real
    #   px, py = vector_pfm(pra, prb, axis_a_deg=0, axis_b_deg=90)
    #   labels = snap_to_axes(px, py, CONFIG["easy_axes_deg"])
    px, py = labels_to_vectors(labels % CONFIG["n_variants"],
                               CONFIG["easy_axes_deg"])
    ITER_STATE["last_scan"] = (h, a, ph)
    return labels, px, py

def run_iteration(params, family, target_labels, target_winding=None,
                  initial_labels=None, speed_ums=None, show=True):
    k = ITER_STATE["k"]; ITER_STATE["k"] += 1
    tb = params_to_builder(family, params)
    fname = os.path.join(CONFIG["work_dir"], f"iter{k:03d}_{family}.txt")
    x_m, y_m, v = tb.save(fname)
    shape = target_labels.shape
    ext = (0, CONFIG["field_um"], 0, CONFIG["field_um"])

    if CONFIG["DRY_RUN"]:
        init = (np.zeros(shape, int) if initial_labels is None else initial_labels)
        labels = simulate_write(x_m, y_m, v, ext, shape, CONFIG["easy_axes_deg"],
                                initial_labels=init,
                                v_coercive=CONFIG["v_coercive_guess"],
                                linewidth_um=CONFIG["linewidth_um_guess"],
                                bias_sign_convention=CONFIG["bias_sign_convention"])
        px, py = labels_to_vectors(labels, CONFIG["easy_axes_deg"])
    else:
        send_trajectory(fname)
        start_trajectory(speed_ums=speed_ums)
        wait_for_litho(tb, speed_ums=speed_ums)
        labels, px, py = measure_labels(shape)

    loss = score_against_target(labels, target_labels, CONFIG["n_variants"],
                                px=px, py=py, target_winding=target_winding)
    if show:
        fig, ax = plt.subplots(1, 2, figsize=(8, 3.6))
        ax[0].imshow(target_labels, origin="lower", cmap="twilight",
                     vmin=-1, vmax=CONFIG["n_variants"]-1); ax[0].set_title("target")
        ax[1].imshow(labels, origin="lower", cmap="twilight",
                     vmin=-1, vmax=CONFIG["n_variants"]-1)
        ax[1].set_title(f"iter {k}: loss={loss:.3f}")
        plt.tight_layout(); plt.show()
    return loss, labels

## 7. Session A — compass-rose calibration → sector map

Write isolated strokes spanning motion angle × bias on a uniform blank slate, read with PFM, and fit the **sector map** `variant → motion angle` (and the bias-sign convention). This is the empirical switching transfer function everything else looks up.

Run order at the microscope:
1. Pole a blank slate (one big `gen_raster_fill` over the field, single direction).
2. Write the rose, rescan, then call `fit_sector_map` on the labeled image.

In [ ]:
# --- A1. blank slate ------------------------------------------------------
fld = CONFIG["field_um"]
blank = gen_raster_fill(np.ones((8, 8), bool), (0, fld, 0, fld),
                        stroke_angle_deg=0.0, pitch_um=0.08,
                        v=-CONFIG["v_write"], step_um=CONFIG["step_um"])
blank.save(os.path.join(CONFIG["work_dir"], "blank_slate.txt"))
visualize_trajectory(tb=blank, title="blank slate poling")
# At the scope:  send_trajectory(...); start_trajectory(); wait_for_litho(blank)

In [ ]:
# --- A2. compass rose -------------------------------------------------------
angles = np.arange(0, 360, 30)
rose, manifest = gen_compass_rose(angles, biases_v=[-CONFIG["v_write"],
                                                    +CONFIG["v_write"]],
                                  line_len_um=0.8, cell_um=1.2, margin_um=0.5,
                                  step_um=CONFIG["step_um"])
rose_file = os.path.join(CONFIG["work_dir"], "rose.txt")
rose.save(rose_file)
print(f"rose field needed: {rose.field_um:.1f} um  "
      f"(set scan/litho frame accordingly)")
visualize_trajectory(tb=rose, field_um=rose.field_um, title="compass rose")

In [ ]:
def fit_sector_map(manifest, labels, field_um, background=None, win_um=0.4):
    """Read the variant written by each rose stroke and build
    {variant: circular-mean motion angle} per bias sign."""
    H, W = labels.shape
    out = []
    for m in manifest:
        ix = int(m["cx_um"] / field_um * W); iy = int(m["cy_um"] / field_um * H)
        w = max(1, int(win_um / field_um * W / 2))
        patch = labels[max(0, iy-w):iy+w+1, max(0, ix-w):ix+w+1].ravel()
        if background is not None:
            patch = patch[patch != background]
        if len(patch) == 0:
            continue
        vals, cnts = np.unique(patch, return_counts=True)
        out.append({**m, "variant": int(vals[np.argmax(cnts)])})
    sector = {}
    for sign in (-1, +1):
        sub = [o for o in out if np.sign(o["V"]) == sign]
        per_var = {}
        for u in sorted({o["variant"] for o in sub}):
            angs = np.deg2rad([o["angle_deg"] for o in sub if o["variant"] == u])
            per_var[u] = float(np.rad2deg(np.angle(np.mean(np.exp(1j*angs)))) % 360)
        sector[sign] = per_var
    return out, sector

# --- A3. measure & fit (dry-run shown; replace with real scan at the scope) ---
shape = (CONFIG["scan_px"], CONFIG["scan_px"])
if CONFIG["DRY_RUN"]:
    x_m, y_m, v = rose.to_arrays(); x_m, y_m = x_m*1e-6, y_m*1e-6
    lab_rose = simulate_write(x_m, y_m, v, (0, rose.field_um, 0, rose.field_um),
                              shape, CONFIG["easy_axes_deg"],
                              initial_labels=np.zeros(shape, int),
                              v_coercive=CONFIG["v_coercive_guess"],
                              linewidth_um=0.3,
                              bias_sign_convention=CONFIG["bias_sign_convention"])
else:
    lab_rose, _, _ = measure_labels(shape)

records, SECTOR = fit_sector_map(manifest, lab_rose, rose.field_um, background=0)
plt.figure(figsize=(4,4)); plt.imshow(lab_rose, origin="lower", cmap="twilight")
plt.title("rose readout (labels)"); plt.show()
print("sector map {bias sign: {variant: motion angle}}:")
print(json.dumps(SECTOR, indent=1))
# Persist the calibrated map for the compiler:
SECTOR_MAP = SECTOR.get(int(np.sign(-CONFIG["v_write"])), SECTOR.get(-1, {}))

**What to extract from Session A before moving on**
* `SECTOR_MAP` — motion angle per variant (compiler input).
* `bias_sign_convention` — does flipping V flip the selected variant by 180°? Update `CONFIG`.
* Repeat the rose at several |V| and speeds → coercive window + written linewidth (`linewidth_um_guess`). Sweep `v_amplitude` cells exactly like the bias-magnitude panel of Checa Fig. 6j–k.
* Repeat on different blank-slate variants → is switching initial-state dependent?

## 8. Session B — compiler test: bicrystal with a designed wall

First real inverse-problem test: two half-planes of chosen variants ⇒ one designed domain wall. Sweep `(v_left, v_right)` over the wall classes (60°/120°, neutral vs charged) to build your wall-stability table.

In [ ]:
shape = (CONFIG["scan_px"], CONFIG["scan_px"])
tgt_bi = target_bicrystal(shape, v_left=0, v_right=2, wall_angle_deg=90)
v_map  = {u: -CONFIG["v_write"] for u in range(CONFIG["n_variants"])}
# geometric default (motion // easy axis, flipped if anti-parallel convention),
# overridden by whatever Session A actually calibrated:
# flip motion by 180 only when sign(V_write) x convention says anti-parallel:
flip = 180 if (np.sign(-CONFIG["v_write"]) * CONFIG["bias_sign_convention"] < 0) else 0
sector_default = {u: (CONFIG["easy_axes_deg"][u] + flip) % 360
                  for u in range(CONFIG["n_variants"])}
sector_map = {**sector_default,
              **{int(k): v for k, v in (SECTOR_MAP or {}).items()}}

loss_bi, lab_bi = run_iteration(
    dict(target_labels=tgt_bi, sector_map=sector_map, v_map=v_map,
         pitch_um=0.06),
    family="compiler", target_labels=tgt_bi)
print("bicrystal loss:", round(loss_bi, 3))

## 9. Session C — active learning toward a topological target

Goal-directed Bayesian optimization: pick a target (here a 6-sector **convergent** structure; swap in `target_vortex(..., radial=False)` for flux closure), optimize spiral/flower parameters with GP-EI. Each `run_iteration` is one write+read cycle (~2–5 min on hardware), so 20–40 iterations is an afternoon.

In [ ]:
shape = (CONFIG["scan_px"], CONFIG["scan_px"])
if CONFIG["DRY_RUN"]:
    # The toy simulator follows a pure motion-direction rule, so a spiral can
    # only paint TANGENTIAL textures -> demo target = flux-closure vortex.
    TARGET = target_vortex(shape, CONFIG["easy_axes_deg"], sense=+1,
                           radial=False, bg=-1, radius_frac=0.4)
else:
    # On hardware, nucleation physics (Checa) makes spirals produce radial
    # convergent/divergent states -> the interesting target:
    TARGET = target_vortex(shape, CONFIG["easy_axes_deg"], sense=-1,
                           radial=True, bg=-1, radius_frac=0.4)  # convergent
TARGET_WINDING = 1.0

SPACE = {
    "turns":     (5, 60),
    "radius_um": (0.4, 0.45 * CONFIG["field_um"]),
    "v":         (-9.0, 9.0),               # sign = polarity (Checa Fig. 3)
    "direction": ["out", "in"],
    "chirality": ["ccw", "cw"],
}
al = ActiveLearner(SPACE, seed=0,
                   log_path=os.path.join(CONFIG["work_dir"], "al_log.json"))

N_SEED, N_BO = 6, 14
for p in al.seed_points(N_SEED):
    loss, _ = run_iteration(p, "spiral", TARGET, target_winding=TARGET_WINDING,
                            show=False)
    al.observe(p, loss, meta={"family": "spiral"})
    print({k: (round(v,2) if isinstance(v,float) else v) for k,v in p.items()},
          "-> loss", round(loss, 3))

for it in range(N_BO):
    p = al.suggest()
    loss, lab = run_iteration(p, "spiral", TARGET, target_winding=TARGET_WINDING,
                              show=(it % 5 == 0))
    al.observe(p, loss, meta={"family": "spiral"})
    print(f"BO {it:02d}", {k: (round(v,2) if isinstance(v,float) else v)
                           for k,v in p.items()}, "-> loss", round(loss, 3))

best_p, best_loss = al.best()
print("\nBEST:", best_p, "loss", round(best_loss, 3))

In [ ]:
# Convergence + rerun of best point with the plot on
plt.figure(figsize=(5,3))
plt.plot(np.minimum.accumulate(al.y), "o-")
plt.xlabel("iteration"); plt.ylabel("best loss so far"); plt.grid(alpha=.3)
plt.title("active-learning convergence"); plt.tight_layout(); plt.show()
_ = run_iteration(best_p, "spiral", TARGET, target_winding=TARGET_WINDING)

### Extending Session C
* **Flux closure**: `TARGET = target_vortex(..., radial=False, sense=±1)`, `family="flower"` with `SPACE = {petals, radius_um, v, chirality, outward_only}` — chirality should matter here (Checa Fig. 5g) but not for spirals (Fig. 6l): a free physics check on your film.
* **Exploration vs exploitation**: interleave `al.seed_points(1)[0]` every ~5 BO steps to keep mapping the space globally.
* **Closed-loop compiler repair**: after a compiler write, diff `labels != target`, re-fill only the failed region (`compile_target` on the diff mask) — greedy model-predictive rewriting.
* **Initial-state dependence**: pass the *measured* previous `labels` as `initial_labels` (dry-run) or simply don't blank-slate between iterations (hardware) and add the previous pattern's features to the GP input.

## 10. Placeholders & to-verify list (search `# >>> PLACEHOLDER`)

1. **`litho_is_running()`** — round-trip of `GV("LithoRunning")` into Python. Until then `wait_for_litho` uses a path-length/speed time estimate (+15 % + margin), which is safe but blind to a stalled run.
2. **Scan-frame setup in `acquire_pfm()`** — programmatic ScanSize / offset / points so the read frame always matches the litho field; currently assumed set manually in the AR software.
3. **Second sensing axis (`acquire_pfm_second_orientation`)** — needed for unambiguous 6-variant maps; 1-channel GMM is the interim.
4. **Drift application (`register_offset`)** — function computes (dx, dy) µm between scans; wire it into `start_trajectory(center_um=...)` or into the analysis extent.
5. **NaN pen-up rows** — `TrajectoryBuilder(nan_breaks=True)` emits NaN segment breaks, but I don't know whether `TL_LoadBuildPy` accepts them; default is zero-bias travel strokes, which is also the physically safer choice.
6. **ibw channel indices** — v1 used `data[0]/[1]/[3]`; confirm for your PFM mode (`CONFIG["chan_*"]`).
7. **Litho-field vs scan-frame origin** — generators use corner origin 0..field (matching v1's spiral file); `TL_RunPy` manual center is assumed to be in scan-frame µm. Verify the mapping once with the overlay (`TL_ToggleOverlay()`).